# V7 — Deep Learning (MLP) on 10,000 synthetic + 10 real movies

**Architecture**: 4-layer MLP with batch normalisation and dropout.

**Targets**: `imdb_rating` and `wom_multiplier_log` (regression).

**Evaluation strategy** (both reported):
1. **5-fold CV on augmented (n=10,010)** — direct comparison with the v6 classical models.
2. **Synth→real transfer** — train on 10,000 synthetic, test on 10 real. **This is the meaningful evaluation** because the Gaussian copula manufactures consistency among synthetic samples by construction; transfer to held-out real data is the only way to test whether DL extracts signal beyond what classical ML does.

**Methodological note**: 10,000 synthetic from 10 real movies is a 1,000:1 augmentation ratio. The copula cannot create signal beyond what is in the 10 real films. CV on the augmented set will be optimistic. Treat the transfer result as the headline.

In [ ]:
# Colab: mount Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    from pathlib import Path
    PROJECT = Path('/content/drive/MyDrive/MSC THESIS')
except Exception:
    from pathlib import Path
    PROJECT = Path('/Users/davidtoma/Library/CloudStorage/GoogleDrive-tomadavid001@gmail.com/My Drive/MSC THESIS')

V6_DIR = PROJECT / 'ml_dataset' / 'data' / 'model_ready' / 'movie_success_v6'
V7_DIR = PROJECT / 'ml_dataset' / 'data' / 'model_ready' / 'movie_success_v7'
RUN_DIR = V7_DIR / 'colab_runs_v7'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('V6 (real):', V6_DIR)
print('V7 (synth):', V7_DIR)
print('Run dir:', RUN_DIR)

Mounted at /content/drive
V6 (real): /content/drive/MyDrive/MSC THESIS/ml_dataset/data/model_ready/movie_success_v6
V7 (synth): /content/drive/MyDrive/MSC THESIS/ml_dataset/data/model_ready/movie_success_v7
Run dir: /content/drive/MyDrive/MSC THESIS/ml_dataset/data/model_ready/movie_success_v7/colab_runs_v7


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr, pearsonr

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| Device:', DEVICE)

PyTorch: 2.10.0+cu128 | Device: cuda


In [ ]:
# Load real (n=10) and synthetic (n=10000) movie-level features
real_mov  = pd.read_csv(V6_DIR / 'movie_features_v6.csv')
real_meta = pd.read_csv(V6_DIR / 'scene_movie_metadata_v6.csv')
syn_mov   = pd.read_csv(V7_DIR / 'movie_features_v7_synthetic.csv')
syn_meta  = pd.read_csv(V7_DIR / 'scene_movie_metadata_v7_synthetic.csv')

print(f'Real movies:     {len(real_mov)} × {real_mov.shape[1]}')
print(f'Synthetic movies: {len(syn_mov)} × {syn_mov.shape[1]}')

Real movies:     10 × 327
Synthetic movies: 10000 × 331


In [ ]:
# Target-leakage exclusions (same as v6)
LEAKAGE = ['budget_usd', 'revenue_usd', 'roi_percent', 'success_class',
           'imdb_rating', 'wom_multiplier', 'wom_multiplier_log']

real_mov = real_mov.drop(columns=LEAKAGE, errors='ignore')
syn_mov  = syn_mov.drop(columns=LEAKAGE, errors='ignore')

META_KEEP = ['movie_id', 'targeted_emotion', 'clip_duration_s',
             'cut_count', 'brightness', 'motion_intensity', 'audio_loudness',
             'silence_ratio', 'music_presence', 'dialogue_density',
             'face_screen_time_ratio', 'lead_screen_time_ratio',
             'release_year', 'genre_primary', 'genre_secondary',
             'country_of_origin', 'budget_categorical',
             'imdb_rating', 'wom_multiplier', 'wom_multiplier_log']

real_meta_sub = real_meta[[c for c in META_KEEP if c in real_meta.columns]]
syn_meta_sub  = syn_meta[[c for c in META_KEEP if c in syn_meta.columns]]

real_mov = real_mov.merge(real_meta_sub, on='movie_id', how='left'); real_mov['is_synthetic']=0
syn_mov  = syn_mov.merge(syn_meta_sub, on='movie_id', how='left');  syn_mov['is_synthetic']=1
df_all = pd.concat([real_mov, syn_mov], ignore_index=True)
print(f'Combined: {len(df_all)} movies')

Combined: 10010 movies


In [ ]:
# Encode metadata (same scheme as v6 notebooks)
DROP = {'movie_id','condition','n_participants',
        'imdb_rating','wom_multiplier','wom_multiplier_log','is_synthetic'}
df_feat = df_all.copy()
ORD = {'low':1,'moderate':2,'high':3}
SCENE_ORDINAL = ['cut_count','brightness','motion_intensity','audio_loudness',
                 'silence_ratio','music_presence','dialogue_density',
                 'face_screen_time_ratio','lead_screen_time_ratio']
for c in SCENE_ORDINAL + ['budget_categorical']:
    if c in df_feat.columns: df_feat[c] = df_feat[c].map(ORD)
OH = [c for c in ['targeted_emotion','genre_primary','genre_secondary','country_of_origin']
      if c in df_feat.columns]
df_feat = pd.get_dummies(df_feat, columns=OH, prefix_sep='_', dummy_na=False, dtype=int)

ALL_FEATS = [c for c in df_feat.columns if c not in DROP]
X_all = df_feat[ALL_FEATS].apply(pd.to_numeric, errors='coerce')
X_all = X_all.drop(columns=X_all.columns[X_all.isna().all()].tolist())
X_all = X_all.drop(columns=X_all.columns[X_all.std()==0].tolist())
y_imdb = df_feat['imdb_rating'].astype(float)
y_wom  = df_feat['wom_multiplier_log'].astype(float)
is_syn = df_feat['is_synthetic'].astype(int).values

print(f'Final feature matrix: {X_all.shape[0]} rows × {X_all.shape[1]} features')
print(f'  Real rows: {(is_syn==0).sum()}  | Synthetic rows: {(is_syn==1).sum()}')

Final feature matrix: 10010 rows × 360 features
  Real rows: 10  | Synthetic rows: 10000


In [ ]:
# MLP architecture
class MLPRegressor(nn.Module):
    def __init__(self, n_features, hidden=(256, 128, 64), dropout=0.3):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_mlp(X_train, y_train, X_val, y_val, n_features, epochs=120, batch_size=128, lr=1e-3, patience=20):
    model = MLPRegressor(n_features).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    Xt = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
    yt = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)
    Xv = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
    yv = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    ds = TensorDataset(Xt, yt)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)

    best_val = float('inf'); best_state = None; bad = 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward(); opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            val_pred = model(Xv)
            val_loss = loss_fn(val_pred, yv).item()
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience: break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        return model(Xv).cpu().numpy()

def metrics(y_true, y_pred):
    return dict(
        r2=float(r2_score(y_true, y_pred)),
        mae=float(mean_absolute_error(y_true, y_pred)),
        rmse=float(np.sqrt(mean_squared_error(y_true, y_pred))),
        spearman=float(spearmanr(y_true, y_pred).correlation),
        pearson=float(pearsonr(y_true, y_pred)[0]),
    )
print('Model + training helpers defined.')

Model + training helpers defined.


## Evaluation 1 — 5-fold CV on augmented set (n=10,010)

This is the augmented-CV result that parallels v6's classical models. **Expected to be optimistic** because the synthetic data dominates each fold and the copula manufactures consistency by construction.

In [ ]:
def cv_5fold(X, y, n_features, seed=42):
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    y_pred = np.zeros(len(y))
    for fold, (tr, te) in enumerate(kf.split(X)):
        # impute + scale on training set only
        imp = SimpleImputer(strategy='median'); imp.fit(X[tr])
        sc  = StandardScaler(); sc.fit(imp.transform(X[tr]))
        Xtr = sc.transform(imp.transform(X[tr]))
        Xte = sc.transform(imp.transform(X[te]))
        y_pred[te] = train_mlp(Xtr, y[tr], Xte, y[te], n_features)
        print(f'  fold {fold+1}/5 done')
    return y_pred

X_np = X_all.values.astype(np.float32)
n_features = X_np.shape[1]

print('IMDb 5-fold CV (MLP)...')
imdb_pred = cv_5fold(X_np, y_imdb.values.astype(np.float32), n_features)
imdb_cv = metrics(y_imdb.values, imdb_pred)
print('IMDb augmented CV:', imdb_cv)

print('\nWOM 5-fold CV (MLP)...')
wom_pred = cv_5fold(X_np, y_wom.values.astype(np.float32), n_features)
wom_cv = metrics(y_wom.values, wom_pred)
print('WOM augmented CV:', wom_cv)

IMDb 5-fold CV (MLP)...
  fold 1/5 done
  fold 2/5 done
  fold 3/5 done
  fold 4/5 done
  fold 5/5 done
IMDb augmented CV: {'r2': 0.6672761459879818, 'mae': 0.28221237775210023, 'rmse': 0.34401253414878297, 'spearman': 0.7516356347139989, 'pearson': 0.860775435931638}

WOM 5-fold CV (MLP)...
  fold 1/5 done
  fold 2/5 done
  fold 3/5 done
  fold 4/5 done
  fold 5/5 done
WOM augmented CV: {'r2': 0.2385070086491924, 'mae': 0.4506453757516154, 'rmse': 0.5739991313038312, 'spearman': 0.4764560325559807, 'pearson': 0.5157330766880085}


## Evaluation 2 — Synth → Real transfer (the meaningful test)

Train on the 10,000 synthetic movies only. Test on the 10 real movies, held out entirely from training. This evaluates whether the MLP extracts signal that **transfers** to the real distribution — the only check that the augmentation isn't producing pure tautology.

In [ ]:
is_real = (is_syn == 0)
is_synth = (is_syn == 1)

X_synth = X_np[is_synth]; X_real = X_np[is_real]
y_imdb_synth = y_imdb.values.astype(np.float32)[is_synth]
y_imdb_real  = y_imdb.values.astype(np.float32)[is_real]
y_wom_synth  = y_wom.values.astype(np.float32)[is_synth]
y_wom_real   = y_wom.values.astype(np.float32)[is_real]

# Split synthetic into train (90%) / val (10%) for early stopping
from sklearn.model_selection import train_test_split
X_tr, X_va, y_tr_imdb, y_va_imdb = train_test_split(X_synth, y_imdb_synth, test_size=0.1, random_state=SEED)
_,    _,    y_tr_wom,  y_va_wom  = train_test_split(X_synth, y_wom_synth,  test_size=0.1, random_state=SEED)

# Preprocess (fit on synthetic train only)
imp = SimpleImputer(strategy='median'); imp.fit(X_tr)
sc  = StandardScaler(); sc.fit(imp.transform(X_tr))
X_tr_p = sc.transform(imp.transform(X_tr))
X_va_p = sc.transform(imp.transform(X_va))
X_real_p = sc.transform(imp.transform(X_real))

n_features = X_tr_p.shape[1]

print('IMDb — train on 10K synthetic, test on 10 real...')
pred_imdb_real = train_mlp(X_tr_p, y_tr_imdb, X_real_p, y_imdb_real, n_features, epochs=200, patience=30)
imdb_transfer = metrics(y_imdb_real, pred_imdb_real)
print('IMDb transfer:', imdb_transfer)

print('\nWOM — train on 10K synthetic, test on 10 real...')
pred_wom_real = train_mlp(X_tr_p, y_tr_wom, X_real_p, y_wom_real, n_features, epochs=200, patience=30)
wom_transfer = metrics(y_wom_real, pred_wom_real)
print('WOM transfer:', wom_transfer)

IMDb — train on 10K synthetic, test on 10 real...
IMDb transfer: {'r2': 0.7614046335220337, 'mae': 0.29282140731811523, 'rmse': 0.34013953436387906, 'spearman': 0.8231860349807907, 'pearson': 0.8727089166641235}

WOM — train on 10K synthetic, test on 10 real...
WOM transfer: {'r2': 0.33097243309020996, 'mae': 0.4522523880004883, 'rmse': 0.49426381079693216, 'spearman': 0.5515151515151515, 'pearson': 0.6129603981971741}


## Save results

In [ ]:
results = {
    'task': 'v7_deep_learning_mlp',
    'architecture': '4-layer MLP (256-128-64-1) with BN, ReLU, Dropout 0.3, AdamW, cosine LR',
    'n_total': int(len(df_all)),
    'n_real':  int((is_syn==0).sum()),
    'n_synth': int((is_syn==1).sum()),
    'n_features': int(n_features),
    'imdb_5fold_cv': imdb_cv,
    'wom_5fold_cv':  wom_cv,
    'imdb_synth_to_real_transfer': imdb_transfer,
    'wom_synth_to_real_transfer':  wom_transfer,
}
out = RUN_DIR / 'results_mlp_v7.json'
with open(out, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved → {out}')
print(json.dumps(results, indent=2))

Saved → /content/drive/MyDrive/MSC THESIS/ml_dataset/data/model_ready/movie_success_v7/colab_runs_v7/results_mlp_v7.json
{
  "task": "v7_deep_learning_mlp",
  "architecture": "4-layer MLP (256-128-64-1) with BN, ReLU, Dropout 0.3, AdamW, cosine LR",
  "n_total": 10010,
  "n_real": 10,
  "n_synth": 10000,
  "n_features": 360,
  "imdb_5fold_cv": {
    "r2": 0.6672761459879818,
    "mae": 0.28221237775210023,
    "rmse": 0.34401253414878297,
    "spearman": 0.7516356347139989,
    "pearson": 0.860775435931638
  },
  "wom_5fold_cv": {
    "r2": 0.2385070086491924,
    "mae": 0.4506453757516154,
    "rmse": 0.5739991313038312,
    "spearman": 0.4764560325559807,
    "pearson": 0.5157330766880085
  },
  "imdb_synth_to_real_transfer": {
    "r2": 0.7614046335220337,
    "mae": 0.29282140731811523,
    "rmse": 0.34013953436387906,
    "spearman": 0.8231860349807907,
    "pearson": 0.8727089166641235
  },
  "wom_synth_to_real_transfer": {
    "r2": 0.33097243309020996,
    "mae": 0.452252388000

## Interpretation

Two numbers matter most:

- **`imdb_synth_to_real_transfer.r2` and `.spearman`** — these test whether the MLP, trained on 10,000 synthetic movies, can predict the 10 real movies' IMDb ratings. If transfer R² is comparable to or better than v6's nested-CV RF (R² = 0.49, ρ = 0.64), the MLP is genuinely competitive. If transfer R² is much lower than the 5-fold-CV R², the MLP is fitting synthetic-specific structure that doesn't generalise.
- **WOM transfer** — expected to be poor, mirroring v6's WOM uniformity (R² ≤ 0.21).

**Caveat for thesis**: 5-fold CV on the augmented set is inflated by construction. Do not report it as the headline. Report transfer.